In [40]:
import pandas as pd
import altair as alt

In [39]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/GRÁFICA/base_datos_Oscar_corregida.csv")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,nombres_nominados_nominadas,genero,lugar_de_nacimiento,nacionalidad_1,nacionalidad_2,raza_o_etnia,su_debut_fue_estadounidense,titulo_original_pelicula_debut,categoria_nominacion_oscar,fue_ganador_o_ganadora,ano_ceremonia,titulo_original_pelicula_nominada,ano_pelicula_nominada,idioma_original,generos,companias_de_produccion,paises_de_produccion,idiomas_hablados
0,Richard Barthelmess,Masculino,"Nueva York, Estados Unidos",Estadounidense,NaN,Blanca,Sí,War Brides,Mejor actor,No,1929,The Noose,1927,Inglés,"Drama, Crimen",First National Pictures,Estados Unidos,Inglés
1,Richard Barthelmess,Masculino,"Nueva York, Estados Unidos",Estadounidense,NaN,Blanca,Sí,War Brides,Mejor actor,No,1929,The Patent Leather Kid,1927,Inglés,"Guerra, Drama, Romance",First National Pictures,Estados Unidos,Inglés
2,Emil Jannings,Masculino,"Rorschach, Suiza",Suiza,Alemana,Blanca,No,Arme Eva,Mejor actor,Sí,1929,The Last Command,1927,Inglés,"Drama, Bélica",Paramount Pictures,Estados Unidos,Inglés
3,Emil Jannings,Masculino,"Rorschach, Suiza",Suiza,Alemana,Blanca,No,Arme Eva,Mejor actor,No,1929,The Way of All Flesh,1927,Inglés,Drama,Paramount,Estados Unidos,Inglés
4,Louise Dresser,Femenino,"Evansville, Estados Unidos",Estadounidense,NaN,Blanca,Sí,The Glory of Clementina,Mejor actriz,No,1929,A Ship Comes In,1927,Inglés,Drama,"DeMille Pictures Corporation, Pathé Exchange",Estados Unidos,Inglés


In [44]:
# Limpiamos los espacios de la columna nacionalidad_1
df["nacionalidad_1"] = df["nacionalidad_1"].astype(str).str.strip()

# Convertimos nacionalidad y debut en V y F
es_estadounidense = df["nacionalidad_1"] == "Estadounidense"
debut_estadounidense = df["su_debut_fue_estadounidense"] == "Sí"
df["es_estadounidense"] = es_estadounidense
df["debut_estadounidense"] = debut_estadounidense

# Definimos la función clasificar
def clasificar(fila):
    if fila["es_estadounidense"] and fila["debut_estadounidense"]:
        return "Estadounidense con debut en EE.UU."
    elif fila["es_estadounidense"]:
        return "Estadounidense con debut internacional"
    elif fila["debut_estadounidense"]:
        return "Extranjero con debut en EE.UU."
    else:
        return "Extranjero con debut internacional"

# Aplicamos la función a cada fila
df["clasificar"] = df.apply(clasificar, axis=1)

# Añadimos la década y agrupamos la operación antes de convertir a string
df["década"] = ((df["ano_ceremonia"] // 10) * 10).astype(str) + "s"

# Contamos las nominaciones por década y clasificación, calculando el porcentaje
t = df.groupby(["década", "clasificar"]).size().reset_index(name="n")
t["total"] = t.groupby("década")["n"].transform("sum")
t["pct"] = t["n"] / t["total"] * 100

t.head()

# Definimos el orden de los colores que queremos asignar a cada categoría
orden = ["Estadounidense con debut en EE.UU.",
         "Estadounidense con debut internacional",
         "Extranjero con debut en EE.UU.",
         "Extranjero con debut internacional"]
colores = ["#963527", "#B5AE9F", "#A68224", "#3D3837"]  # Código colores (HEX)

# Gráfico interactivo + tooltip
gráfico = alt.Chart(t).mark_bar().encode(
    x=alt.X("n:Q", stack="normalize",
            title="Porcentaje de nominadas y nominados a los Oscar",
            axis=alt.Axis(format="%")),
    y=alt.Y("década:O", title="Décadas"),
    color=alt.Color("clasificar:N", title="Clasificación", sort=orden,
                    scale=alt.Scale(domain=orden, range=colores),
                    legend=alt.Legend(
                    labelLimit=400,    # Límite del ancho para los márgenes
                    symbolSize=100,    # Tamaño cuadritos de colores
)),
    tooltip=[
        alt.Tooltip("n:Q", title="Nominaciones"),
        alt.Tooltip("total", title="Total de nominaciones década"),
        alt.Tooltip("pct:Q", title="Porcentaje", format=".2f")
]
# Propiedades del gráfico
).properties(
    width=650,
    height=400,
    title="Clasificación de nominaciones a los Oscar por década"
)

gráfico

alt.Chart(...)

In [45]:
gráfico.save("gráfico.html")

In [46]:
from google.colab import files
files.download("grafico.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
%pip install vl-convert-python --quiet

In [55]:
from google.colab import files
from PIL import Image

# 1. Guardamos el gráfico en alta resolución como PNG
gráfico.save("temp_gráfico.png", scale_factor=3.0)

# 2. Abrimos la imagen y la convertimos a JPG (RGB)
imagen_png = Image.open("temp_gráfico.png")
imagen_jpg = imagen_png.convert("RGB")

# 3. Guardamos la imagen en JPG con buena calidad
imagen_jpg.save("gráfico.jpg", "JPEG", quality=95)

# 4. Descargamos el archivo
files.download("gráfico.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>